In [17]:
import os
import joblib
import numpy as np
import pandas as pd
import xgboost as xgb
from scipy.stats import spearmanr

folder_path = r"D:\OneDrive\Trading\Market Making\data\runs\run_20260607_064103"

snapshots = pd.read_parquet(os.path.join(folder_path, "snapshots.parquet"))

snapshots

,ts,symbol,best_bid,best_ask,mid,microprice,best_bid_tick,best_ask_tick,mid_tick,spread,...,ask_delta,quote_churn,future_mid_100ms,future_return_100ms,future_mid_500ms,future_return_500ms,future_mid_1000ms,future_return_1000ms,future_mid_5000ms,future_return_5000ms
0,1780713724914,BTCUSDT,60980.06,60980.07,60980.065,60980.060331,6098006,6098007,6098006,0.01,...,0.0,0.0,60980.065,0.0,60980.065,0.0,60980.065,0.0,60964.005,-0.000263
1,1780713725014,BTCUSDT,60980.06,60980.07,60980.065,60980.060331,6098006,6098007,6098006,0.01,...,0.0,0.0,60980.065,0.0,60980.065,0.0,60980.065,0.0,60964.005,-0.000263
2,1780713725114,BTCUSDT,60980.06,60980.07,60980.065,60980.060338,6098006,6098007,6098006,0.01,...,0.0,0.0,60980.065,0.0,60980.065,0.0,60980.065,0.0,60964.005,-0.000263
3,1780713725214,BTCUSDT,60980.06,60980.07,60980.065,60980.060374,6098006,6098007,6098006,0.01,...,0.0,0.0,60980.065,0.0,60980.065,0.0,60980.065,0.0,60964.005,-0.000263
4,1780713725314,BTCUSDT,60980.06,60980.07,60980.065,60980.060372,6098006,6098007,6098006,0.01,...,0.0,0.0,60980.065,0.0,60980.065,0.0,60980.065,0.0,60964.005,-0.000263
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20514,1780715776314,BTCUSDT,60883.43,60883.44,60883.435,60883.431305,6088343,6088344,6088344,0.01,...,0.0,0.0,60883.435,0.0,NaN,NaN,NaN,NaN,NaN,NaN
20515,1780715776414,BTCUSDT,60883.43,60883.44,60883.435,60883.430575,6088343,6088344,6088344,0.01,...,0.0,0.0,60883.435,0.0,NaN,NaN,NaN,NaN,NaN,NaN
20516,1780715776514,BTCUSDT,60883.43,60883.44,60883.435,60883.430575,6088343,6088344,6088344,0.01,...,0.0,0.0,60883.435,0.0,NaN,NaN,NaN,NaN,NaN,NaN
20517,1780715776615,BTCUSDT,60883.43,60883.44,60883.435,60883.431305,6088343,6088344,6088344,0.01,...,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [23]:
"""

Does ml_delta explain the residual error in my reservation price (mid + struct_delta + micro_signal) | market state?

"""

df = snapshots

horizons = [100, 500, 1000, 5000]

df["reservation_y"] = df["mid"] + df["struct_delta"] + df["micro_signal_delta"] # reconstructed reservation, just to be clear what goes in reservation

for h in horizons:
    df[f"y_{h}ms"] = np.log(df[f"future_mid_{h}ms"] / df["reservation_y"]) # residuals with center = mid + struct_delta + micro_drift

feature_cols = [
    # raw microstructure
    "spread",
    "order_imbalance",
    "trade_imbalance",
    "inventory",
    "volatility",
    "queue_ahead_bid",
    "queue_ahead_ask",
]

df = df.dropna()

split = int(len(df) * 0.8)
train = df.iloc[:split]
test = df.iloc[split:]

X_train = train[feature_cols].to_numpy(dtype=np.float32)
X_test = test[feature_cols].to_numpy(dtype=np.float32)

In [24]:
def ic(pred, y):
    return np.corrcoef(pred, y)[0, 1]

def rank_ic(pred, y):
    return spearmanr(pred, y).statistic

results = {}
models = {}

for h in horizons:

    y_train = train[f"y_{h}ms"]
    y_test = test[f"y_{h}ms"]

    model = xgb.XGBRegressor(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42
    )

    model.fit(X_train, y_train)

    train_pred = model.predict(X_train)

    pred = model.predict(X_test)
    true = y_test.values

    # -------------------------
    # signal quality
    # -------------------------
    residual_ic = ic(pred, true)
    residual_rank_ic = rank_ic(pred, true)

    # -------------------------
    # direction accuracy
    # -------------------------
    hit_rate = (np.sign(pred) == np.sign(true)).mean()

    # -------------------------
    # true economic interpretation
    # -------------------------
    pnl_proxy = np.mean(pred * true)
    pnl_std = np.std(pred * true) + 1e-9
    sharpe_proxy = pnl_proxy / pnl_std

    models[h] = {
        "model": model
    }
    results[h] = {
        "Residual_IC": residual_ic,
        "Residual_Rank_IC": residual_rank_ic,
        "HitRate": hit_rate,
        "PnLProxy": pnl_proxy,
        "SharpeProxy": sharpe_proxy,
    }

results_df = pd.DataFrame(results)
results_df

c:\Users\admin\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\numpy\lib\_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
c:\Users\admin\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\numpy\lib\_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
C:\Users\admin\AppData\Local\Temp\ipykernel_44208\664318283.py:5: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(pred, y).statistic


,100,500,1000,5000
Residual_IC,NaN,NaN,2.052910e-01,1.526334e-01
Residual_Rank_IC,NaN,NaN,-5.036679e-02,1.421449e-01
HitRate,5.004885e-01,4.892526e-01,3.187592e-01,5.276014e-01
PnLProxy,-1.709867e-13,-1.827612e-12,1.144115e-10,3.176412e-09
SharpeProxy,-1.690259e-04,-1.718627e-03,7.424556e-02,1.530433e-01


In [28]:
artifact = {
    "model": models[1000]["model"],
    "feature_cols": feature_cols,
    "target": "log(future_mid/(mid + struct_delta + micro_signal_delta))",
    "horizon_ms": 1000,
}

joblib.dump(artifact, "data/residual_model_3.pkl")

['data/residual_model_3.pkl']